In [1]:
import subprocess, os, pandas as pd

# ─────────────────────────────────────────────
# CELL 1 — Install all tools
# ─────────────────────────────────────────────

os.makedirs("/kaggle/tmp", exist_ok=True)          # ensure tmp exists for scratch
os.makedirs("/kaggle/working/tools", exist_ok=True)

# ── sratoolkit ──────────────────────────────
print("Installing sratoolkit...")
subprocess.run(["wget", "-q",
    "https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz",
    "-O", "/kaggle/working/sratoolkit.tar.gz"], check=True)
subprocess.run(["tar", "-xzf", "/kaggle/working/sratoolkit.tar.gz",
    "-C", "/kaggle/working/tools/"], check=True)
sra_dir = [d for d in os.listdir("/kaggle/working/tools/")
           if d.startswith("sratoolkit") and os.path.isdir(f"/kaggle/working/tools/{d}")][0]
SRA_BIN = f"/kaggle/working/tools/{sra_dir}/bin"
os.remove("/kaggle/working/sratoolkit.tar.gz")
print(f"  sratoolkit: {SRA_BIN}")

# ── STAR ────────────────────────────────────
print("Installing STAR...")
subprocess.run(["wget", "-q",
    "https://github.com/alexdobin/STAR/releases/download/2.7.11b/STAR_2.7.11b.zip",
    "-O", "/kaggle/working/STAR.zip"], check=True)
subprocess.run(["unzip", "-q", "/kaggle/working/STAR.zip",
    "-d", "/kaggle/working/tools/"], check=True)
STAR = "/kaggle/working/tools/STAR_2.7.11b/Linux_x86_64_static/STAR"
subprocess.run(["chmod", "+x", STAR], check=True)
os.remove("/kaggle/working/STAR.zip")
r = subprocess.run([STAR, "--version"], capture_output=True, text=True)
print(f"  STAR: {r.stdout.strip()}")

# ── featureCounts ───────────────────────────
print("Installing featureCounts...")
subprocess.run(["wget", "-q",
    "https://sourceforge.net/projects/subread/files/subread-2.0.6/subread-2.0.6-Linux-x86_64.tar.gz/download",
    "-O", "/kaggle/working/subread.tar.gz"], check=True)
subprocess.run(["tar", "-xzf", "/kaggle/working/subread.tar.gz",
    "-C", "/kaggle/working/tools/"], check=True)
FC = "/kaggle/working/tools/subread-2.0.6-Linux-x86_64/bin/featureCounts"
os.remove("/kaggle/working/subread.tar.gz")
print(f"  featureCounts installed: {os.path.exists(FC)}")

print("\nAll tools ready.")

Installing sratoolkit...
  sratoolkit: /kaggle/working/tools/sratoolkit.3.3.0-ubuntu64/bin
Installing STAR...
  STAR: 2.7.11b
Installing featureCounts...
  featureCounts installed: True

All tools ready.


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Load STAR index + GTF from dataset
# ─────────────────────────────────────────────
import os, subprocess, shutil

GENOME_DIR   = "/kaggle/temp/genome"
INDEX        = "/kaggle/temp/star_index"
GTF          = f"{GENOME_DIR}/annotation.gtf"
DATASET_PATH = "/kaggle/input/datasets/feyddautha/star-index-grch38-109/star_index"

os.makedirs(GENOME_DIR, exist_ok=True)

# ── Load STAR index from dataset ──────────────────────────────
if not os.path.exists(f"{INDEX}/SA"):
    print("Copying STAR index from dataset into /kaggle/temp...")
    subprocess.run(["cp", "-r", DATASET_PATH, "/kaggle/temp/"], check=True)
    print("  Index ready.")
else:
    print("STAR index already in /kaggle/temp — skipping.")

# ── Download GTF (not in dataset, small ~50 MB) ───────────────
if not os.path.exists(GTF):
    print("Downloading GTF...")
    subprocess.run(["wget", "-q",
        "https://ftp.ensembl.org/pub/release-109/gtf/homo_sapiens/"
        "Homo_sapiens.GRCh38.109.gtf.gz",
        "-O", f"{GENOME_DIR}/annotation.gtf.gz"], check=True)
    subprocess.run(["gunzip", "-f", f"{GENOME_DIR}/annotation.gtf.gz"], check=True)
    print("  GTF ready.")
else:
    print("GTF already exists — skipping.")

# ── Sanity check ──────────────────────────────────────────────
assert os.path.exists(f"{INDEX}/SA"),  "ERROR: SA file missing from index!"
assert os.path.exists(GTF),            "ERROR: GTF not found!"

d = shutil.disk_usage("/kaggle/temp")
print(f"\n/kaggle/temp free: {d.free/1e9:.1f} GB / {d.total/1e9:.1f} GB")
print(f"INDEX : {INDEX}")
print(f"GTF   : {GTF}")
print("\nCell 2 complete — ready for alignment.")

Copying STAR index from dataset into /kaggle/temp...
  Index ready.
  GTF ready.

/kaggle/temp free: 1399.1 GB / 8656.9 GB
INDEX : /kaggle/temp/star_index
GTF   : /kaggle/temp/genome/annotation.gtf

Cell 2 complete — ready for alignment.


In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Process all samples (SAFE + STABLE)
# ─────────────────────────────────────────────

import os, shutil, subprocess, pandas as pd, time, gc

# ─────────────────────────────────────────────
# Safe remove with retry
# ─────────────────────────────────────────────
def safe_remove_retry(path, retries=3, delay=2):
    for i in range(retries):
        try:
            if os.path.isdir(path):
                shutil.rmtree(path)
            elif os.path.exists(path):
                os.remove(path)
            return
        except Exception as e:
            if i < retries - 1:
                time.sleep(delay)
            else:
                print(f"Warning: failed to remove {path}: {e}")

# ─────────────────────────────────────────────
# SRR lists
# ─────────────────────────────────────────────
SRR_LIST_ALL = [
    "SRR36692421", "SRR36692422", "SRR36692423", "SRR36692424",
    "SRR36692425", "SRR36692426", "SRR36692427", "SRR36692428",
    "SRR36692429", "SRR36692430", "SRR36692431", "SRR36692432",
    "SRR36692433", "SRR36692434", "SRR36692435", "SRR36692436",
    "SRR36692437", "SRR36692438", "SRR36692439", "SRR36692440",
    "SRR36692441", "SRR36692442", "SRR36692443", "SRR36692444",
    "SRR36692445", "SRR36692446", "SRR36692447", "SRR36692448",
    "SRR36692449", "SRR36692450", "SRR36692451", "SRR36692452",
    "SRR36692453", "SRR36692454", "SRR36692455", "SRR36692456",
    "SRR36692457", "SRR36692458", "SRR36692459", "SRR36692460",
    "SRR36692461", "SRR36692462", "SRR36692463", "SRR36692464",
    "SRR36692465", "SRR36692466", "SRR36692467", "SRR36692468",
    "SRR36692469", "SRR36692470", "SRR36692471", "SRR36692472",
    "SRR36692473", "SRR36692474",
]

SRR_LIST = [
    #"SRR36692421",
    "SRR36692422", "SRR36692423", "SRR36692424",
    "SRR36692425", "SRR36692426", "SRR36692427", "SRR36692428",
    "SRR36692429", "SRR36692430", "SRR36692431", "SRR36692432",
]

# ─────────────────────────────────────────────
# Paths
# ─────────────────────────────────────────────
INDEX        = "/kaggle/temp/star_index"
GTF          = "/kaggle/temp/genome/annotation.gtf"
COUNT_MATRIX = "/kaggle/working/count_matrix.csv"

failed = []
all_counts = {}

# ─────────────────────────────────────────────
# Resume checkpoint
# ─────────────────────────────────────────────
if os.path.exists(COUNT_MATRIX):
    existing = pd.read_csv(COUNT_MATRIX, index_col=0)
    already_done = list(existing.columns)
    all_counts = existing.to_dict()
    print(f"Resuming — {len(already_done)} samples already done")
else:
    already_done = []

# ─────────────────────────────────────────────
# Disk monitor
# ─────────────────────────────────────────────
def disk_free_gb(path="/kaggle/temp"):
    return round(shutil.disk_usage(path).free / 1e9, 1)

# ─────────────────────────────────────────────
# Main loop
# ─────────────────────────────────────────────
for i, SRR in enumerate(SRR_LIST, 1):

    print(f"\n{'='*60}")
    print(f"  [{i}/{len(SRR_LIST)}]  {SRR}   (disk free: {disk_free_gb()} GB)")
    print(f"{'='*60}")

    if SRR in already_done:
        print("  Already done — skipping.")
        continue

    try:
        # ─────────────────────────────
        # 1. prefetch
        # ─────────────────────────────
        print("  [1/4] prefetch...")
        r = subprocess.run([
            f"{SRA_BIN}/prefetch", SRR, "-O", "/kaggle/temp/"
        ], capture_output=True, text=True)

        if r.returncode != 0:
            raise RuntimeError(f"prefetch failed: {r.stderr[-500:]}")

        time.sleep(2)

        # ─────────────────────────────
        # 2. fasterq-dump
        # ─────────────────────────────
        print("  [2/4] fasterq-dump...")
        r = subprocess.run([
            f"{SRA_BIN}/fasterq-dump",
            "--split-files",
            "--temp",   "/kaggle/temp",
            "--outdir", "/kaggle/temp",
            f"/kaggle/temp/{SRR}"
        ], capture_output=True, text=True)

        # remove SRA directory safely
        safe_remove_retry(f"/kaggle/temp/{SRR}")

        if r.returncode != 0:
            raise RuntimeError(f"fasterq-dump failed: {r.stderr[-500:]}")

        time.sleep(2)

        fq1 = f"/kaggle/temp/{SRR}_1.fastq"
        fq2 = f"/kaggle/temp/{SRR}_2.fastq"

        if not os.path.exists(fq1):
            raise RuntimeError("fastq files not found after fasterq-dump")

        # ─────────────────────────────
        # 3. STAR alignment
        # ─────────────────────────────
        print("  [3/4] STAR alignment...")
        align_dir = f"/kaggle/temp/align_{SRR}"
        os.makedirs(align_dir, exist_ok=True)

        r = subprocess.run([
            STAR,
            "--runThreadN",        "4",
            "--genomeDir",         INDEX,
            "--readFilesIn",       fq1, fq2,
            "--outSAMtype",        "BAM", "SortedByCoordinate",
            "--outFileNamePrefix", f"{align_dir}/",
            "--outSAMattributes",  "NH", "HI", "AS", "NM",
        ], capture_output=True, text=True)

        # remove FASTQ immediately
        safe_remove_retry(fq1)
        safe_remove_retry(fq2)

        if r.returncode != 0:
            raise RuntimeError(f"STAR failed: {r.stderr[-500:]}")

        time.sleep(2)

        bam = f"{align_dir}/Aligned.sortedByCoord.out.bam"
        if not os.path.exists(bam):
            raise RuntimeError("BAM not produced by STAR")

        # ─────────────────────────────
        # 4. featureCounts
        # ─────────────────────────────
        print("  [4/4] featureCounts...")
        counts_file = f"/kaggle/temp/counts_{SRR}.txt"

        r = subprocess.run([
            FC, "-T", "4", "-p",
            "-a", GTF, "-o", counts_file, bam
        ], capture_output=True, text=True)

        # remove BAM directory safely
        safe_remove_retry(align_dir)

        if r.returncode != 0:
            raise RuntimeError(f"featureCounts failed: {r.stderr[-500:]}")

        time.sleep(2)

        # ─────────────────────────────
        # Parse counts
        # ─────────────────────────────
        counts_df = pd.read_csv(counts_file, sep="\t", comment="#", index_col=0)
        all_counts[SRR] = counts_df.iloc[:, -1].to_dict()

        pd.DataFrame(all_counts).to_csv(COUNT_MATRIX)

        safe_remove_retry(counts_file)

        gc.collect()

        print(f"  ✓ Done  (disk free: {disk_free_gb()} GB)")

    except Exception as e:
        print(f"  ✗ FAILED: {e}")
        failed.append(SRR)

        # ─────────────────────────────
        # Cleanup on failure
        # ─────────────────────────────
        safe_remove_retry(f"/kaggle/temp/{SRR}")
        safe_remove_retry(f"/kaggle/temp/align_{SRR}")

        for fname in os.listdir("/kaggle/temp"):
            if fname.startswith(SRR):
                safe_remove_retry(f"/kaggle/temp/{fname}")

        gc.collect()
        time.sleep(2)

# ─────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"COMPLETE — {len(all_counts)} processed, {len(failed)} failed")

if failed:
    print(f"Failed: {failed}")

final = pd.read_csv(COUNT_MATRIX, index_col=0)
print(f"Matrix shape: {final.shape}  (genes × samples)")


  [1/11]  SRR36692422   (disk free: 1399.1 GB)
  [1/4] prefetch...
  [2/4] fasterq-dump...
  [3/4] STAR alignment...
